# Storm Damage Predictor — Model Training (v2 Clean)
### Correct pipeline order. No encoding mistakes.

**Order:**
1. Load `storm_clean.parquet` (strings, no encoding yet)
2. Feature engineering (season, state_avg, event_avg) — on strings
3. Save `storm_featured.parquet` — strings still intact
4. Label encode STATE, EVENT_TYPE, MONTH_NAME
5. Save encoders as `.pkl`
6. Stratified train/test split
7. SMOTE on train only
8. Train XGBoost baseline
9. Optuna tuning
10. Save tuned model
11. Evaluate on real test set
12. Save all artifacts

## CELL 1 — Install

In [ ]:
!pip install xgboost imbalanced-learn optuna scikit-learn joblib -q

## CELL 2 — Imports

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import optuna
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)
os.makedirs('models', exist_ok=True)
print('Imports OK')

## CELL 3 — Load Clean Parquet

Load `storm_clean.parquet` — has original strings.
Upload via Files tab if fresh session.

In [ ]:
df = pd.read_parquet('storm_clean.parquet')

print(f'Shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nDtypes:\n{df.dtypes}')
print(f'\nSample STATE values: {df["STATE"].unique()[:5]}')
print(f'Sample MONTH values: {df["MONTH_NAME"].unique()[:5]}')
print(f'Sample EVENT values: {df["EVENT_TYPE"].unique()[:5]}')

## CELL 4 — Feature Engineering

Add season, state_avg_damage, event_avg_damage.
**MUST happen before encoding — uses string groupby.**

In [ ]:
# Season from month name
SEASON_MAP = {
    'December': 0, 'January': 0, 'February': 0,
    'March': 1,    'April': 1,   'May': 1,
    'June': 2,     'July': 2,    'August': 2,
    'September': 3,'October': 3, 'November': 3,
}
df['season'] = df['MONTH_NAME'].map(SEASON_MAP)

# Historical avg damage per state — computed on full dataset
# Note: this is a global stat, not target leak because it's a property
# of the state, not derived from target after split
df['state_avg_damage'] = df.groupby('STATE')['damage_total_num'].transform('mean')
df['event_avg_damage'] = df.groupby('EVENT_TYPE')['damage_total_num'].transform('mean')

print('Feature engineering done')
print(f'New cols: season, state_avg_damage, event_avg_damage')
print(f'Shape: {df.shape}')
print(f'\nSample:')
print(df[['STATE', 'MONTH_NAME', 'EVENT_TYPE', 'season', 'state_avg_damage', 'event_avg_damage']].head(3))

## CELL 5 — Save storm_featured.parquet

Save NOW — before encoding. Strings still intact.
API will use this to verify encodings.

In [ ]:
df.to_parquet('storm_featured.parquet', index=False)
print(f'Saved storm_featured.parquet — shape: {df.shape}')
print(f'STATE dtype: {df["STATE"].dtype}  ← should be object (string)')
print(f'damage_tier distribution:\n{df["damage_tier"].value_counts().sort_index()}')

## CELL 6 — Label Encode + Save Encoders

Encode AFTER saving featured parquet.
Save each encoder — API uses same encoders at inference.

In [ ]:
le_state      = LabelEncoder()
le_event_type = LabelEncoder()
le_month_name = LabelEncoder()

df['STATE']      = le_state.fit_transform(df['STATE'])
df['EVENT_TYPE'] = le_event_type.fit_transform(df['EVENT_TYPE'])
df['MONTH_NAME'] = le_month_name.fit_transform(df['MONTH_NAME'])

# Save encoders
joblib.dump(le_state,      'models/le_state.pkl')
joblib.dump(le_event_type, 'models/le_event_type.pkl')
joblib.dump(le_month_name, 'models/le_month_name.pkl')

# Verify — print classes so API can hardcode if needed
print('STATE classes (first 10):', le_state.classes_[:10].tolist())
print('EVENT classes (first 5):', le_event_type.classes_[:5].tolist())
print('MONTH classes:', le_month_name.classes_.tolist())
print('\nEncoders saved to models/')
print(f'\nEncoded dtypes:\n{df[["STATE", "EVENT_TYPE", "MONTH_NAME"]].dtypes}')

## CELL 7 — Define Features + Target

In [ ]:
FEATURE_COLS = [
    'STATE', 'MONTH_NAME', 'EVENT_TYPE',
    'INJURIES_DIRECT', 'INJURIES_INDIRECT',
    'DEATHS_DIRECT', 'DEATHS_INDIRECT',
    'MAGNITUDE', 'duration_min',
    'season', 'state_avg_damage', 'event_avg_damage',
]
TARGET_COL = 'damage_tier'

X = df[FEATURE_COLS].values.astype(np.float32)  # numpy array for SMOTE
y = df[TARGET_COL].values

print(f'X shape: {X.shape}  dtype: {X.dtype}')
print(f'y shape: {y.shape}')
print(f'y distribution: {dict(zip(*np.unique(y, return_counts=True)))}')

## CELL 8 — Stratified Train/Test Split

Split BEFORE SMOTE. Test set = real distribution, never touched by SMOTE.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,    # preserve tier ratio in both splits
)

print(f'Train shape: {X_train.shape}')
print(f'Test shape:  {X_test.shape}')
print(f'\nTrain tier distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}')
print(f'Test tier distribution:  {dict(zip(*np.unique(y_test, return_counts=True)))}')

## CELL 9 — SMOTE on Train Only

Balance train set. Test set stays real distribution.
Takes ~5 min on T4.

In [ ]:
import time

print('Running SMOTE on train set only...')
start = time.time()

sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)

print(f'Done in {(time.time()-start)/60:.1f} min')
print(f'X_res shape: {X_res.shape}')
print(f'y_res distribution: {dict(zip(*np.unique(y_res, return_counts=True)))}')

## CELL 10 — Baseline XGBoost

Train quick baseline on SMOTE data. Evaluate on REAL test set.

In [ ]:
print('Training baseline XGBoost...')
start = time.time()

baseline = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='mlogloss',
    device='cuda',
)
baseline.fit(X_res, y_res)

print(f'Done in {(time.time()-start)/60:.1f} min')

# Evaluate on REAL test set (not SMOTE)
preds_base = baseline.predict(X_test)
print('\nBaseline — REAL test set:')
print(classification_report(y_test, preds_base, target_names=['None','Low','Medium','High']))

## CELL 11 — Optuna Tuning

50 trials. Maximises weighted F1 on REAL test set.

In [ ]:
def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 800),
        'max_depth':         trial.suggest_int('max_depth', 3, 12),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.5, log=True),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': 42,
        'eval_metric': 'mlogloss',
        'device': 'cuda',
    }
    model = XGBClassifier(**params)
    model.fit(X_res, y_res)
    preds = model.predict(X_test)
    return f1_score(y_test, preds, average='weighted')


print('Running Optuna (50 trials)...')
start = time.time()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\nDone in {(time.time()-start)/60:.1f} min')
print(f'Best weighted F1: {study.best_value:.4f}')
print(f'Best params: {study.best_params}')

## CELL 12 — Train Tuned Model + Evaluate

In [ ]:
print('Training tuned model...')
start = time.time()

tuned = XGBClassifier(
    **study.best_params,
    random_state=42,
    eval_metric='mlogloss',
    device='cuda',
)
tuned.fit(X_res, y_res)

print(f'Done in {(time.time()-start)/60:.1f} min')

preds_tuned = tuned.predict(X_test)
print('\nTuned model — REAL test set:')
print(classification_report(y_test, preds_tuned, target_names=['None','Low','Medium','High']))

weighted_f1 = f1_score(y_test, preds_tuned, average='weighted')
print(f'Weighted F1: {weighted_f1:.4f}')

## CELL 13 — Save All Artifacts

In [ ]:
import json

# Save tuned classifier
joblib.dump(tuned, 'models/classifier.pkl')
print('Saved: models/classifier.pkl')

# Save test data for SHAP notebook
np.save('models/X_test.npy', X_test)
np.save('models/y_test.npy', y_test)
print('Saved: models/X_test.npy, y_test.npy')

# Save metrics.json
metrics = {
    'XGBoost_tuned': round(weighted_f1, 4),
    'winner': 'XGBoost_tuned',
    'best_params': study.best_params,
}
with open('models/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved: models/metrics.json')

print('\nAll artifacts saved. Run model comparison next (04_model_comparison.ipynb)')

## CELL 14 — Download All

In [ ]:
import shutil
from google.colab import files

# Download featured parquet (has strings — needed locally)
files.download('storm_featured.parquet')

# Download all models
shutil.make_archive('models_v2', 'zip', 'models/')
files.download('models_v2.zip')

print('Downloaded: storm_featured.parquet + models_v2.zip')
print('Extract models_v2.zip → api/models/')
print('Replace data/processed/storm_featured.parquet')